# 🐍 AI Travel Agent with Microsoft Agent Framework (Python) - Custom Model Compatible

## 📋 Scenario Overview

This notebook demonstrates how to build an intelligent travel planning agent using the Microsoft Agent Framework for Python. The agent leverages your custom LLM (Tencent Cloud, Deepseek or other OpenAI-compatible API) to automatically generate personalized day-trip itineraries for random destinations worldwide.

**Key Features:**
- 🎲 **Smart Destination Selection**: Custom tool function for random destination picking
- 🗺️ **Detailed Itinerary Generation**: AI-powered travel planning with local recommendations
- 🔄 **Async Processing**: Uses asyncio for efficient API communication
- 🛠️ **Tool Integration**: Demonstrates function calling capabilities in AI agents

## 🏗️ Technical Implementation

### Core Components
- **Agent Framework**: Python implementation of Microsoft's agent orchestration system
- **Custom Models API**: Access to state-of-the-art language models via your OpenAI-compatible API
- **OpenAI Compatibility**: Uses OpenAI client patterns with custom model backend
- **Environment Management**: Secure credential handling with python-dotenv

### Architecture Flow
```python
User Request → ChatAgent → Custom Model API ↔ get_random_destination()
                     ↓
              Travel Itinerary Response
```

### Key Classes & Methods
- `ChatAgent`: Main conversational agent orchestrator
- `OpenAIChatClient`: Custom Models API client wrapper
- `get_random_destination()`: Custom tool function for destination selection
- Environment variables: Secure API configuration management

## ⚙️ Prerequisites & Setup

**Required Dependencies:**
```bash
pip install agent-framework-core -U
```

**Environment Configuration (.env file):**
```env
OPENAI_API_KEY=your_custom_api_key
OPENAI_ENDPOINT=https://your-custom-model-endpoint.com/v1
OPENAI_CHAT_MODEL_ID=your_preferred_model
# For compatibility with existing code:
GITHUB_TOKEN=$OPENAI_API_KEY
GITHUB_ENDPOINT=$OPENAI_ENDPOINT
GITHUB_MODEL_ID=$OPENAI_CHAT_MODEL_ID
```

## 🚀 Usage Instructions

Execute the cells below in sequence to:
1. Import required libraries and load environment variables
2. Define the random destination generator tool
3. Create and configure the AI agent
4. Run travel planning requests and view results

Let's build an intelligent travel planning assistant! 🌟

In [ ]:
! pip install agent-framework-core -U

In [ ]:
# 📦 Import Required Libraries
# Standard library imports for system operations and random number generation
import os
from random import randint

# Third-party library for loading environment variables from .env file
from dotenv import load_dotenv

# Import model adapter
from model_adapter import get_openai_client

In [ ]:
# 🤖 Import Microsoft Agent Framework Components
# ChatAgent: The main agent class for conversational AI
# OpenAIChatClient: Client for connecting to OpenAI-compatible APIs (including custom models)
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient

In [ ]:
# 🔧 Load Environment Variables
# This loads configuration from a .env file in the project root
# Required variables: OPENAI_ENDPOINT, OPENAI_API_KEY, OPENAI_CHAT_MODEL_ID (mapped to GITHUB_*)
load_dotenv()

# Verify that all required environment variables are set
required_vars = ['OPENAI_API_KEY', 'OPENAI_ENDPOINT', 'OPENAI_CHAT_MODEL_ID']
for var in required_vars:
    if not os.getenv(var):
        print(f'⚠️  Warning: {var} is not set in environment')
        
# Ensure GITHUB_* variables are also available (for compatibility with existing code)
os.environ.setdefault('GITHUB_TOKEN', os.getenv('OPENAI_API_KEY', ''))
os.environ.setdefault('GITHUB_ENDPOINT', os.getenv('OPENAI_ENDPOINT', ''))
os.environ.setdefault('GITHUB_MODEL_ID', os.getenv('OPENAI_CHAT_MODEL_ID', ''))

In [ ]:
# 🎲 Tool Function: Random Destination Generator
# This function will be available to the agent as a tool
# The agent can call this function to get random vacation destinations
def get_random_destination() -> str:
    """Get a random vacation destination.
    
    Returns:
        str: A randomly selected destination from our predefined list
    """
    # List of popular vacation destinations around the world
    destinations = [
        "Barcelona, Spain",
        "Paris, France", 
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    ]
    # Return a random destination from the list
    return destinations[randint(0, len(destinations) - 1)]

In [ ]:
# 🔗 Create OpenAI Chat Client for Custom Models
# This client connects to your custom model API (OpenAI-compatible endpoint)
# Environment variables required:
# - GITHUB_ENDPOINT: API endpoint URL (your custom model endpoint)
# - GITHUB_TOKEN: Your API key
# - GITHUB_MODEL_ID: Model to use (your preferred model)
try:
    openai_chat_client = OpenAIChatClient(
        base_url=os.environ.get("GITHUB_ENDPOINT"),
        api_key=os.environ.get("GITHUB_TOKEN"), 
        model_id=os.environ.get("GITHUB_MODEL_ID")
    )
    print(f'✅ Successfully connected to model: {os.environ.get("GITHUB_MODEL_ID")}')
    print(f'   Endpoint: {os.environ.get("GITHUB_ENDPOINT")}')
except Exception as e:
    print(f'❌ Error creating OpenAI client: {e}')
    print('   Make sure your .env file is properly configured with OPENAI_* variables')

In [ ]:
# 🤖 Create the Travel Planning Agent
# This creates a conversational AI agent with specific capabilities:
# - chat_client: The AI model client for generating responses
# - instructions: System prompt that defines the agent's personality and role
# - tools: List of functions the agent can call to perform actions
try:
    agent = ChatAgent(
        chat_client=openai_chat_client,
        instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations.",
        tools=[get_random_destination]  # Our random destination tool function
    )
    print('✅ Agent created successfully!')
except Exception as e:
    print(f'❌ Error creating agent: {e}')

In [ ]:
# 🚀 Run the Agent
# Send a message to the agent and get a response
# The agent will use its tools (get_random_destination) if needed
try:
    response = await agent.run("Plan me a day trip")
    print('✅ Agent executed successfully!')
except Exception as e:
    print(f'❌ Error running agent: {e}')

In [ ]:
# 📋 View Raw Response Object
# This shows the complete response structure including metadata
# Useful for debugging and understanding the response format
response

In [ ]:
# 📖 Extract and Display the Travel Plan
# Get the last message from the conversation (agent's response)
try:
    last_message = response.messages[-1]
    # Extract the text content from the message
    text_content = last_message.contents[0].text
    # Display the formatted travel plan
    print("🏖️ Travel plan:")
    print(text_content)
except Exception as e:
    print(f'❌ Error extracting response: {e}')
    if 'response' in locals():
        print(f'Raw response: {response}')